# Notebook 03: Quality control, sample by sample

**Goal.** Remove low-quality cells using thresholds computed **inside each sample** (median absolute deviation), so one bad mouse cannot distort the others.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** Loads each raw cohort and adds three quality numbers to every cell: **total_counts** (UMIs), **n_genes** (genes detected), **pct_mt** (percent of reads from mitochondrial genes; a high value means a dying cell). Use *species='human'* for human data (mitochondrial genes start with MT-).

*You should see:* Counts of cells and a describe() table: median UMI about 1,600, median genes about 740.

In [ ]:
from mlc.dataset import Dataset
from mlc import qc, plots
dsets = {}
for c in cohorts()["mouse"]:
    dsets[c] = qc.add_qc(Dataset.load(PROC / f"{c}_raw"), species="mouse")
    print(c, dsets[c].n_obs, "cells in", dsets[c].obs.sample_id.nunique(), "samples")
print(dsets[cohorts()["mouse"][0]].obs[["total_counts", "n_genes", "pct_mt"]].describe().round(1))

**Cell 3.** **mad_filter** removes, within each sample, cells whose UMI count or gene count is more than 5 median-absolute-deviations from that sample's median, cells with more than 15% mitochondrial reads, and cells with fewer than 200 genes. The log (cells removed per sample) is saved: reviewers like to see it. For single-nucleus data use *mt_cap=5*.

*You should see:* About 98-99% kept per cohort (simulated data are cleaner than real data; real cohorts often keep 85-95%).

In [ ]:
qcd, logs = {}, []
for c, ds in dsets.items():
    f, log = qc.mad_filter(ds, nmads=5, mt_cap=15, min_genes=200)
    log["cohort"] = c; logs.append(log); qcd[c] = f
    print(c, "kept", f.n_obs, "of", ds.n_obs, f"({100 * f.n_obs / ds.n_obs:.1f}%)")
logs = pd.concat(logs); logs.to_csv(RES / "qc_log.csv", index=False); print(logs.head(5))

**Cell 4.** Plots the three quality measures for every sample before (grey) and after (teal) filtering. After filtering the long low-quality tails should be gone while the bulk of cells is untouched. If a whole sample looks very different from the others, note it in the paper's supplement.

In [ ]:
c0 = cohorts()["mouse"][0]
plots.fig_qc(dsets[c0].obs, qcd[c0].obs); plt.show()

**Cell 5.** Saves the filtered cohorts so the next notebook can continue from here. Every notebook saves a checkpoint, so you can stop and restart any time without repeating earlier steps.

In [ ]:
for c, f in qcd.items():
    f.save(PROC / f"{c}_qc")
print("saved:", [p.name for p in PROC.glob("*_qc")])